# Phase 3: Multimodal Feature Engineering & Data Loader (Person A)

**Phase:** III (Predictive Modeling)  
**Track Policy:** Tracks 8, 10, 14 (Dev) | Track 21 (SEALED)  

## 📌 Purpose
This notebook serves as the interactive testing and execution environment for the Phase 3 feature pipeline (`scripts/phase3_data_loader.py`). It handles the ingestion, strict mathematical scaling, and sequential windowing of the multimodal dataset to prepare it for deep learning architectures (e.g., 1D CNNs or LSTMs).

**Key Operations:**
1. **Schema Freezing:** Isolates only the approved thermal and SEM feature columns.
2. **Leakage Prevention:** Applies a `StandardScaler` fitted *strictly* on the training tracks using a Leave-One-Track-Out (LOTO) split. This ensures validation and sealed test tracks remain completely unseen by the scaler.
3. **Thermal History Windowing:** Converts flat tabular rows into rolling 3D sequential windows to capture the physical heat buildup and diffusion of the laser over time.

## 🚀 Expected Outputs
Executing this pipeline successfully generates the foundational input structures for the predictive models:
* **`X_seq` (Numpy Arrays):** 3D sequential feature arrays mapped to the shape `(Samples, Window_Size, Features)` for both training and validation splits.
* **`meta_df` (DataFrames):** Aligned metadata containing the `track_id`, `frame_index`, and `x_position_mm` corresponding to the *last* frame of every rolling window. 

*(Note: The `meta_df` output acts as the handshake for Person B to cleanly align the Phase 2 geometric targets and construct the final PyTorch `DataLoader`.)*

In [1]:
# Enable auto-reloading of external modules
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np

# Add the parent directory (project root) to the system path
sys.path.append(os.path.abspath('..'))

# Now Python can find the 'scripts' folder!
from scripts.phase3_data_loader import FeaturePreprocessor

print("✅ FeaturePreprocessor imported successfully!")

✅ FeaturePreprocessor imported successfully!


In [3]:
# Initialize the class
preprocessor = FeaturePreprocessor()

# Define tracks (Track 21 stays sealed!)
train_tracks = [8, 10]
val_tracks = [14]

# IMPORTANT: Make sure this path points to where your CSV actually lives
csv_path = 'C:/Users/adity/Documents/Coursework/Projects/nsf-fmrg-data-challenge/processed_data/final_multimodal_dataset.csv'
#csv_path = "/Users/rick/Desktop/TAMU/NSF Challenge/processed_data/final_multimodal_dataset.csv"

# 1. Run the loader and scaler
train_data, val_data = preprocessor.load_and_scale(
    csv_path=csv_path, 
    train_tracks=train_tracks, 
    eval_tracks=val_tracks
)

# Visually inspect the scaled dataframe
display(train_data.head())

Loading dataset from: C:/Users/adity/Documents/Coursework/Projects/nsf-fmrg-data-challenge/processed_data/final_multimodal_dataset.csv
Filtered dataset to 610 valid rows (pca_ready == True).
Train split: 431 rows (Tracks: [8, 10])
Eval split:  179 rows (Tracks: [14])
✅ Features successfully scaled.


c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\scripts\phase3_data_loader.py:70: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 1.5177605   1.41245548  0.5041997   0.47787344 -0.08814103 -0.33824045
  0.17512151  0.45154719  1.10970355  1.29398734 -0.62782925 -0.28558794
  0.17512151  0.43838406  0.39889468  0.18828464  0.45154719  0.16195839
  1.04388792  0.39889468  1.78102305 -0.20660918  0.22777402  1.30715046
 -0.00916227  0.80695162  0.72797286  1.30715046  1.14919293  0.95174603
  0.63583097  0.49103657  0.54368908 -0.21977231  1.07021417  0.66215722
  0.06981649  1.14919293  0.14879526  0.67532035  0.89909352 -1.37812751
 -0.50936111 -0.39089296  0.85960413  0.17512151  0.42522093  0.51736282
 -0.41721922  1.47827112  0.59634159  0.13563213  0.71480973  1.14919293
  0.30675279  0.74113599  0.30675279  0.9385829   0.4120578   1.04388792
  0.75429912  0.58317846 -0.87792867 -0.37772983 

,track_id,frame_index,x_position_mm,peak_temp,mean_temp,mp_area_px,mp_centroid_x,mp_centroid_y,mp_length,mp_width,...,substrate_side_finite_fraction,baseline_support_count,fallback_baseline_required,is_within_boundary_exclusion,shape_support_fraction_on_common_grid,retained_pca_grid_finite_fraction,normalization_status,shape_center_um_median,shape_scale_um_mad,_descriptor_matched
13,8,350,22.7,1.517761,1.255474,0.929743,0.585872,0.094023,0.952334,0.866973,...,0.549342,167.0,False,False,0.988636,1.0,ok,141.918706,1.260028,True
16,8,353,23.3,1.412455,0.213958,0.862053,0.536028,-1.203855,0.856882,0.850250,...,0.276316,84.0,False,False,0.965909,1.0,ok,136.438786,1.975224,True
17,8,354,23.5,0.504200,0.982928,0.837685,1.644826,-0.525793,0.744571,0.917988,...,0.365132,111.0,False,False,0.965909,1.0,ok,134.500839,1.323030,True
21,8,358,24.3,0.477873,0.313197,-0.007080,-0.339464,4.109646,-0.094278,0.114279,...,0.250000,76.0,False,False,0.960227,1.0,ok,125.230833,1.403972,True
29,8,366,25.9,-0.088141,-0.494101,1.195086,-1.071703,-0.182608,1.053747,1.261592,...,0.398026,121.0,False,False,0.931818,1.0,ok,110.934726,1.749548,True


In [4]:
# 2. Convert to sequential rolling windows
window_size = 5

X_train_seq, train_meta = preprocessor.create_sequence_windows(train_data, window_size=window_size)
X_val_seq, val_meta = preprocessor.create_sequence_windows(val_data, window_size=window_size)

# Verify the 3D shapes (Samples, Timesteps, Features)
print(f"X_train_seq shape: {X_train_seq.shape}")
print(f"X_val_seq shape:   {X_val_seq.shape}")


Creating sequential windows (Size: 5 frames)...
Generated 423 sequences of shape (423, 5, 9) (Samples, Window Size, Features)

Creating sequential windows (Size: 5 frames)...
Generated 175 sequences of shape (175, 5, 9) (Samples, Window Size, Features)
X_train_seq shape: (423, 5, 9)
X_val_seq shape:   (175, 5, 9)


## JUST FOR DEBUGGING BY NABARUN. PLS IGNORE

In [5]:
print(X_train_seq.shape)
print(len(train_meta))
print(train_meta.head(10))

print()

print(X_val_seq.shape)
print(len(val_meta))
print(val_meta.head(10))

(423, 5, 9)
423
   track_id  frame_index  x_position_mm
0       8.0        366.0           25.9
1       8.0        367.0           26.1
2       8.0        369.0           26.5
3       8.0        370.0           26.7
4       8.0        375.0           27.7
5       8.0        378.0           28.3
6       8.0        379.0           28.5
7       8.0        380.0           28.7
8       8.0        381.0           28.9
9       8.0        382.0           29.1

(175, 5, 9)
175
   track_id  frame_index  x_position_mm
0      14.0        452.0           33.5
1      14.0        453.0           33.7
2      14.0        454.0           33.9
3      14.0        456.0           34.3
4      14.0        459.0           34.9
5      14.0        462.0           35.5
6      14.0        463.0           35.7
7      14.0        466.0           36.3
8      14.0        469.0           36.9
9      14.0        471.0           37.3


In [2]:
# ----pls ignore----
import pandas as pd
df = pd.read_csv("C:/Users/adity/Documents/Coursework/Projects/nsf-fmrg-data-challenge/processed_data/final_multimodal_dataset.csv", nrows=0)
print("Available columns:", df.columns.tolist())

Available columns: ['track_id', 'frame_index', 'x_position_mm', 'peak_temp', 'mean_temp', 'mp_area_px', 'mp_centroid_x', 'mp_centroid_y', 'mp_length', 'mp_width', 'sem_tile_index', 'x_start_mm', 'x_end_mm', 'substrate_roughness_variance', 'substrate_mean_intensity', 'heightmap_x_mm', 'heightmap_x_delta_mm', 'heightmap_x_index', 'is_within_heightmap_x_coverage', 'pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'amplitude_um', 'signed_elevation_um', 'eligible', 'nonflat', 'pca_ready', 'regime_id', 'finite_fraction', 'central_corridor_finite_fraction', 'substrate_side_finite_fraction', 'baseline_support_count', 'fallback_baseline_required', 'is_within_boundary_exclusion', 'shape_support_fraction_on_common_grid', 'retained_pca_grid_finite_fraction', 'normalization_status', 'shape_center_um_median', 'shape_scale_um_mad', '_descriptor_matched']


In [4]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Setup Paths
def find_repo_root(start_path: Path = Path.cwd()) -> Path:
    for p in [start_path, *start_path.parents]:
        if (p / "src").exists() or (p / ".git").exists() or (p / "pyproject.toml").exists():
            return p
    return start_path

try:
    REPO_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    REPO_ROOT = find_repo_root()

SRC_DIR = REPO_ROOT / "src"
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from scripts.phase3_data_loader import FeaturePreprocessor
from ml.targets import Phase3TargetAligner

def flatten_windows(X_seq):
    if len(X_seq) == 0:
        return X_seq
    samples, window_size, features = X_seq.shape
    return X_seq.reshape(samples, window_size * features)

def generate_evaluation_artifacts():
    csv_path = REPO_ROOT / "processed_data" / "final_multimodal_dataset.csv"
    output_dir = REPO_ROOT / "processed_data" / "evaluation_plots"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    development_tracks = [8, 10, 14]
    TARGET_GROUP = 'smoothed_macro_width'
    
    print("\n" + "="*60)
    print("PHASE 5: GENERATING VISUALIZATION ARTIFACTS FOR LOTO")
    print("="*60)
    
    all_eval_results = []
    
    fig, axes = plt.subplots(len(development_tracks), 1, figsize=(10, 4 * len(development_tracks)), sharex=False)
    if len(development_tracks) == 1:
        axes = [axes]

    for idx, val_track in enumerate(development_tracks):
        train_tracks = [t for t in development_tracks if t != val_track]
        print(f"\nProcessing LOTO Fold -> Holding out Track {val_track}...")
        
        preprocessor = FeaturePreprocessor()
        train_df, eval_df = preprocessor.load_and_scale(
            csv_path=str(csv_path),
            train_tracks=train_tracks,
            eval_tracks=[val_track]
        )
        
        X_train_seq, train_meta = preprocessor.create_sequence_windows(train_df, window_size=5)
        X_val_seq, val_meta = preprocessor.create_sequence_windows(eval_df, window_size=5)
        
        X_train = flatten_windows(X_train_seq)
        X_val = flatten_windows(X_val_seq)
        
        aligner = Phase3TargetAligner(dataset_path=csv_path)
        Y_train = aligner.align(meta_df=train_meta, target_group=TARGET_GROUP, return_metadata=False).ravel()
        Y_val = aligner.align(meta_df=val_meta, target_group=TARGET_GROUP, return_metadata=False).ravel()
        
        # Train Bayesian Ridge
        model = BayesianRidge()
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_val)
        
        # Collect evaluation metrics per fold
        mae = mean_absolute_error(Y_val, Y_pred)
        rmse = np.sqrt(mean_squared_error(Y_val, Y_pred))
        r2 = r2_score(Y_val, Y_pred)
        
        # Store results for spatial plotting dataframe
        eval_track_df = val_meta.copy()
        eval_track_df['Y_true'] = Y_val
        eval_track_df['Y_pred'] = Y_pred
        eval_track_df['fold'] = val_track
        all_eval_results.append(eval_track_df)
        
        # Plotting Spatial Signal: Actual vs Predicted along x_position_mm
        ax = axes[idx]
        ax.plot(eval_track_df['x_position_mm'], eval_track_df['Y_true'], label='Ground Truth (Smoothed Macro-Width)', color='black', alpha=0.7, linewidth=2)
        ax.plot(eval_track_df['x_position_mm'], eval_track_df['Y_pred'], label=f'Bayesian Ridge Prediction (R²={r2:.3f})', color='tab:blue', linestyle='--', linewidth=2)
        ax.set_title(f"Leave-One-Track-Out Evaluation: Held-Out Track {val_track} (MAE: {mae:.3f}, RMSE: {rmse:.3f})")
        ax.set_xlabel("X Position (mm)")
        ax.set_ylabel("Track Scale / Width (um)")
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='upper right')

    plt.tight_layout()
    spatial_plot_path = output_dir / "loto_spatial_predictions_comparison.png"
    plt.savefig(spatial_plot_path, dpi=300)
    print(f"\n✅ Saved spatial trajectory plot to: {spatial_plot_path}")
    plt.close()

    # Generate parity/scatter plot across all development folds pooled
    pooled_df = pd.concat(all_eval_results, ignore_index=True)
    plt.figure(figsize=(6, 6))
    plt.scatter(pooled_df['Y_true'], pooled_df['Y_pred'], alpha=0.4, color='tab:purple', edgecolors='k', s=25)
    
    min_val = min(pooled_df['Y_true'].min(), pooled_df['Y_pred'].min())
    max_val = max(pooled_df['Y_true'].max(), pooled_df['Y_pred'].max())
    plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='-', linewidth=1.5, label='Ideal Fit (y = x)')
    
    overall_r2 = r2_score(pooled_df['Y_true'], pooled_df['Y_pred'])
    overall_mae = mean_absolute_error(pooled_df['Y_true'], pooled_df['Y_pred'])
    
    plt.title(f"Pooled LOTO Parity Plot (Overall R²={overall_r2:.3f}, MAE={overall_mae:.3f})")
    plt.xlabel("Actual Ground Truth")
    plt.ylabel("Model Predictions")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper left')
    plt.tight_layout()
    
    parity_plot_path = output_dir / "loto_pooled_parity_plot.png"
    plt.savefig(parity_plot_path, dpi=300)
    print(f"✅ Saved parity plot to: {parity_plot_path}")
    plt.close()
    
    print("\n🎉 All evaluation visualization assets successfully compiled and saved under processed_data/evaluation_plots/!")

if __name__ == "__main__":
    generate_evaluation_artifacts()


PHASE 5: GENERATING VISUALIZATION ARTIFACTS FOR LOTO

Processing LOTO Fold -> Holding out Track 8...
Loading dataset from: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\final_multimodal_dataset.csv
Train split: 800 rows (Tracks: [10, 14])
Eval split:  400 rows (Tracks: [8])
✅ Physics features successfully engineered and scaled.

Creating sequential windows (Size: 5 frames)...
Generated 792 sequences of shape (792, 5, 3) (Samples, Window Size, Features)

Creating sequential windows (Size: 5 frames)...
Generated 396 sequences of shape (396, 5, 3) (Samples, Window Size, Features)

Processing LOTO Fold -> Holding out Track 10...
Loading dataset from: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\final_multimodal_dataset.csv
Train split: 800 rows (Tracks: [8, 14])
Eval split:  400 rows (Tracks: [10])
✅ Physics features successfully engineered and scaled.

Creating sequential windows (Size: 5 frames)...
Generated 79

c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\scripts\phase3_data_loader.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 9.78736500e-01  3.18232408e-01  1.24754630e+00  8.86573138e-01
  5.40960532e-01  1.24754630e+00  7.94409776e-01  4.79518291e-01
  1.04017874e+00  1.11698154e+00  6.40804173e-01  4.25756330e-01
  5.79361932e-01  1.60851947e+00  5.71681652e-01  1.57011807e+00
  4.64157730e-01  7.09926695e-01  5.33280252e-01  9.78736500e-01
  7.02246415e-01  8.63532298e-01  7.94409776e-01  8.02090056e-01
  7.56008376e-01  6.63845014e-01  6.25443613e-01  4.94878851e-01
  7.79049216e-01  5.64001372e-01  1.70068283e+00  1.64626805e-01
  3.18232408e-01  5.33280252e-01  7.02246415e-01  7.71368936e-01
  9.40335099e-01  8.71212578e-01  4.41116890e-01  7.94409776e-01
  8.40491457e-01  5.33280252e-01  5.33280252e-01  5.40960532e-01
  8.02090056e-01  9.71056219e-01  5.17919691e-01  5.02559131e-01


Processing LOTO Fold -> Holding out Track 14...
Loading dataset from: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\final_multimodal_dataset.csv
Train split: 800 rows (Tracks: [8, 10])
Eval split:  400 rows (Tracks: [14])
✅ Physics features successfully engineered and scaled.

Creating sequential windows (Size: 5 frames)...
Generated 792 sequences of shape (792, 5, 3) (Samples, Window Size, Features)

Creating sequential windows (Size: 5 frames)...
Generated 396 sequences of shape (396, 5, 3) (Samples, Window Size, Features)

✅ Saved spatial trajectory plot to: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\evaluation_plots\loto_spatial_predictions_comparison.png
✅ Saved parity plot to: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\evaluation_plots\loto_pooled_parity_plot.png

🎉 All evaluation visualization assets successfully compiled and saved under processed_data/evaluat